# Add a model end to end

TorchPP is built so you can drop in your own model and have it train,
sample, and evaluate through the exact same interface as a built-in one.

Every intensity-based model subclasses `IntensityBasedTPP` and implements
`_intensities_at_queries` — the raw intensity $\lambda(t)$ at arbitrary query
times. The base class also needs the cumulative intensity $\Lambda$. Give it in
one of two ways:

- a closed-form `_closed_form_ci` (what the default `CIConfig(mode="closed_form")`
  expects; fast and exact), or
- numerical integration, by overriding `_default_ci_config()` to return
  `CIConfig(mode="numerical", placement="interval", rule="trapezoidal", n_points=50)`.

From these the base class derives the likelihood, the survival function,
sampling, and every predictive query.

This notebook is the runnable companion to `docs/extending.md`. We build a
homogeneous Poisson process — constant intensity — as a user-defined model,
then fit and evaluate it exactly like a built-in.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchpp import IntensityBasedTPP, MVEventData, evaluate_tpp, fit
from torchpp.training import default_fit_config
from torchpp.utils import format_metrics, inverse_softplus

## Define the model

Below is the whole model. Three methods do the work:

- `_intensities_at_queries(query_times, ts)` — the intensity hook.
  Given query times of shape `(B, Q)`, return the raw per-type intensity of
  shape `(B, Q, D)`. For a Poisson the rate is constant, so we broadcast one
  rate per type.
- `transform_params()` — maps the stored *unconstrained* parameters to the
  constrained ones the model uses. We store `raw_mu` and apply `softplus` to
  keep the rates positive.
- `_closed_form_ci(ts, T_low, T)` — the closed-form cumulative intensity
  $\Lambda = \int \lambda$ over the window, which the default closed-form
  `CIConfig` uses. (Without it, override `_default_ci_config()` to integrate
  numerically, as described above.) `_effective_T` folds in the terminating
  per-type time limit.

In [ ]:
class ConstantIntensityTPP(IntensityBasedTPP):
    """A homogeneous Poisson process, written as a user-defined model."""

    def __init__(self, D, terminating=False):
        super().__init__(D=D, terminating=terminating)  # closed-form CI by default
        # Store *unconstrained* parameters; transform_params() applies softplus to
        # enforce positivity. inverse_softplus seeds them so softplus(raw) = 0.1.
        self.raw_mu = nn.Parameter(inverse_softplus(torch.full((D,), 0.1)))

    def transform_params(self):
        return (F.softplus(self.raw_mu),)  # (D,)

    def _intensities_at_queries(self, query_times, ts):
        # Raw intensity at (B, Q) query times -> (B, Q, D).
        (mu,) = self.transform_params()
        B, Q = query_times.shape
        return mu.view(1, 1, -1).expand(B, Q, self.D)

    def _closed_form_ci(self, ts, T_low, T):
        # Closed-form Lambda over [T_low or 0, T] (used by the default CIConfig).
        # _effective_T folds in the terminating per-type limit min(T, t_term_d).
        (mu,) = self.transform_params()
        hi = self._effective_T(T, ts)  # (B, D)
        lo = self._effective_T(T_low, ts) if T_low is not None else 0.0
        return mu.unsqueeze(0) * (hi - lo)  # (B, D)

## Make synthetic data

To have something to fit, we generate homogeneous multivariate Poisson
sequences: for each type, inter-arrival times are exponential with the given
rate. Because we know the true rates, we can later check what the model
recovers. Each sequence is wrapped in an `MVEventData`; the training set is
collated into one padded `BatchedMVEventData`.

In [ ]:
def make_poisson_sequences(n, rates, T, seed):
    """Homogeneous multivariate Poisson via exponential inter-arrival times."""
    g = torch.Generator().manual_seed(seed)
    seqs = []
    for _ in range(n):
        times, types = [], []
        for d, rate in enumerate(rates):
            t = 0.0
            while True:
                t += float(-torch.log(torch.rand((), generator=g)) / rate)
                if t > T:
                    break
                times.append(t)
                types.append(d)
        seqs.append(MVEventData(torch.tensor(times), torch.tensor(types), sort=True, T_obs=T))
    return seqs


TRUE_RATES = [1.5, 0.8]
D, T = len(TRUE_RATES), 20.0
train = make_poisson_sequences(256, TRUE_RATES, T, seed=0)
val = make_poisson_sequences(64, TRUE_RATES, T, seed=1)
test = make_poisson_sequences(64, TRUE_RATES, T, seed=2)

## Fit it

`fit` trains any `TemporalPointProcess`: it minimises `model.loss(batch)` (the per-sequence negative
log-likelihood) per scored event, plus `model.regularization(batch)`, with mini-batches and
validation. Nothing here is specific to our custom class.

In [ ]:
model = ConstantIntensityTPP(D=D)
config = default_fit_config(num_steps=300, learning_rate=0.05, eval_freq=100, batch_size=64)
model = fit(model, train, val, config).model

## Check the fit

The recovered rates should land close to the true `[1.5, 0.8]`. Then we score
the held-out test set with `evaluate_tpp` — the same evaluation entry point
used for built-in models — and print it with `format_metrics`. The point is
that a model defined in a handful of lines trains and evaluates through the
identical uniform API.

`ll_per_event` is the full-sequence log-likelihood per scored event (the training objective). It also contains the probability of no further event between the last event and the end of the window, so it is a bit lower than `joint_ll`, which splits exactly into `marginal_time_ll` + `cond_type_ll`.

To use the model by name with `create_model`, `save_model`/`load_model` and `train.py`, register it
with `torchpp.register_model`; see [`docs/extending.md`](../docs/extending.md).

In [ ]:
with torch.no_grad():
    (fitted_rates,) = model.transform_params()
print("true rates  :", TRUE_RATES)
print("fitted rates:", [round(float(r), 3) for r in fitted_rates])
print("\n" + format_metrics(evaluate_tpp(model, test, compute_marginals=False)))